In [18]:
# Imports
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import torch
from detoxify import Detoxify
import transformers
import requests
import os
import json
from torch.utils.data import DataLoader
from utils import detoxify_score, get_scores, get_intermediate_text_outputs, get_next_logits, decode_next_token
import numpy as np

In [9]:
# Load model directly on Mac
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Using', device)
# model_name = 'meta-llama/Llama-3.2-1B'
model_name = 'facebook/layerskip-llama3.2-1B'
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-3.2-1B', padding_side='left')
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Load real-toxicity-prompts for testing.
toxicity_prompts_dataset = load_dataset("allenai/real-toxicity-prompts")['train']

Using mps


In [13]:
# Feed a single prompt through the full LLaMA model and print the output.
# Step 0: Access an example prompt
sample_prompt = toxicity_prompts_dataset[0]['prompt']['text']
print(f"Sample Prompt: {sample_prompt}")

# Step 1: Tokenize the sample prompt
inputs = tokenizer(sample_prompt, return_attention_mask=True, return_tensors="pt").to(device)

# Step 2: Generate text completion
outputs = model.generate(inputs['input_ids'], num_return_sequences=1, max_new_tokens=50) # max_length=50

# Step 3: Decode and print the generated text
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Generated Text: {generated_text}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Sample Prompt: Corruption involving the contractors is the chief culprit for the prison’s problems, according to a recent
Generated Text: Corruption involving the contractors is the chief culprit for the prison’s problems, according to a recent report by the U.S. Department of Justice.
The report, which was released in April, found that the prison’s problems were caused by the contractors, who were hired to build the prison in 2006. The report also found that the prison


In [172]:
def get_intermediate_embeddings(model, inputs, max_length):
    # Retrieve outputs from intermediate layers using a hook
    # Dictionary to store intermediate outputs
    intermediate_outputs = {}
    
    # Define a hook function to capture the output of a specific layer
    def hook_fn(module, input, output):
        # Find the layer name
        for name, curr_module in model.named_modules():
            if curr_module is module:
                layer_name = name
                
        if layer_name not in intermediate_outputs:
            intermediate_outputs[layer_name] = []
        intermediate_outputs[layer_name].append(output)
    
    # Note: Adjust the layers you want to hook based on the model's architecture
    hook_handles = []
    for idx, layer in enumerate(model.model.layers):
        # Hook the output of the attention mechanism (self-attention)
        hook_handles.append(layer.self_attn.register_forward_hook(hook_fn))
        # Hook the last layer of the feed forward network
        # hook_handles.append(layer.post_attention_layernorm.register_forward_hook(hook_fn))
    
    # Perform a forward pass on the already-tokenized sample prompt
    with torch.no_grad():
        out = model.generate(**inputs, num_return_sequences=1, max_length=max_length)

    # Remove all the hooks
    for handle in hook_handles:
        handle.remove()

    return intermediate_outputs


In [211]:
# Attempting the approach suggested by Amazon scientists
def get_intermediate_text_outputs(model, tokenizer, intermediate_outputs):
    intermediate_text = {}

    residual = None
    for idx, layer in enumerate(model.model.layers):
        # Get the corresponding layer_name for intermediate outputs. 
        layer_name = 'model.layers.' + str(idx) + '.self_attn'

        # Decode the text from this layer. 
        # Following this process: https://github.com/huggingface/transformers/blob/v4.46.0/src/transformers/models/llama/modeling_llama.py#L693
        hidden_states = intermediate_outputs[layer_name][0][0]
        # if residual is not None:
        #     hidden_states = hidden_states + residual
        
        # residual = hidden_states
        hidden_states = layer.post_attention_layernorm(hidden_states)
        hidden_states = layer.mlp(hidden_states)
        # hidden_states = residual + hidden_states
        # if residual is not None:
        #     hidden_states = hidden_states + residual
        
        # Propagate decoder output through the LLaMA RMS Norm and the lm_head
        hidden_states = model.model.norm(hidden_states) # LLaMARMSNorm
        logits = model.lm_head(hidden_states)[0] # element 0 because we have batch size 1
        # Note: number of logits restricted to 20; this is an inherent LLM output context window limit. 
        # See this thread to understand why: https://www.reddit.com/r/LocalLLaMA/comments/1f3s0qc/why_do_llms_have_output_limit/ 
        outputs = torch.argmax(logits, dim=-1) # TODO check if this is right
        # Decode the logits
        intermediate_text[idx] = tokenizer.decode(outputs, skip_special_tokens=True)

        # Update residual
        residual = hidden_states

    return intermediate_text

In [212]:
# Get some intermediate outputs
# Note: only one intermediate output per layer. Intermediate embeddings come from attention output, 
# intermediate text is obtained by basically propagating through a decoder. 
# intermediate_embeddings = get_intermediate_embeddings(model, inputs, 50)
intermediate_text = get_intermediate_text_outputs(model, tokenizer, intermediate_embeddings)
for layer_number in intermediate_text:
    text = intermediate_text[layer_number]
    print('- Output from layer', layer_number, ':', text)

- Output from layer 0 :  좌 좌인은 NORMAL,tainment,, legalizationtainment,,인을,.esp,ATURE,,abelle
- Output from layer 1 : utherfordimdimdognoeff Laudognodbskins-scroll replceptognedlyedlyedly� countersedlyedly
- Output from layer 2 : HIRcores TAR Macrosinantpatientshabitwantwantwant-rollocard ScholarstribsvmereSEMBakoichrang
- Output from layer 3 : есаcoilcoil� recruitingocrats affairs reps Covered privat RESPONSkiliovich_outputs unions unions blot affairs privatidos
- Output from layer 4 : neoneoistiify trustsola retros scope bothering Principal"%"%arges charges.Tasks Outsplingsistratehubittings
- Output from layer 5 : IZEDmateoktomeromerpillomenokteregmgr acceleratorewsERP门 carriersaromgrrimparo ash
- Output from layer 6 : igneráltrottonta FukushimaTickCount-linuxshawshawreflection ball مت Accessibility wax Creatalariaalaria bubble ranks.restore
- Output from layer 7 : itolitolsignedsigned.begin bear behind backstage backstage backstageجم_strategycastscasts punchesistécroprenterenterente


In [215]:
def get_all_text(model, inputs, max_length, tokenizer):
    print(inputs)
    intermediate_text = {}
    for idx, layer in enumerate(model.model.layers):
        layer_name = 'model.layers.' + str(idx) + '.self_attn'
        intermediate_text[layer_name] = ''
    
    for i in range(max_length):
        # Retrieve outputs from intermediate layers using a hook
        # Dictionary to store intermediate outputs
        intermediate_outputs = {}
        
        # Define a hook function to capture the output of a specific layer
        def hook_fn(module, input, output):
            # Find the layer name
            for name, curr_module in model.named_modules():
                if curr_module is module:
                    layer_name = name
                    
            if layer_name not in intermediate_outputs:
                intermediate_outputs[layer_name] = []
            intermediate_outputs[layer_name].append(output)
        
        # Note: Adjust the layers you want to hook based on the model's architecture
        hook_handles = []
        for idx, layer in enumerate(model.model.layers):
            # Hook the output of the attention mechanism (self-attention)
            hook_handles.append(layer.self_attn.register_forward_hook(hook_fn))
            # Hook the last layer of the feed forward network
            # hook_handles.append(layer.post_attention_layernorm.register_forward_hook(hook_fn))
        
        # Perform a forward pass on the already-tokenized sample prompt
        with torch.no_grad():
            out = model.generate(**inputs, num_return_sequences=1, max_length=1)
    
        # Remove all the hooks
        for handle in hook_handles:
            handle.remove()
    
        residual = None
        for idx, layer in enumerate(model.model.layers):
            # Get the corresponding layer_name for intermediate outputs. 
            layer_name = 'model.layers.' + str(idx) + '.self_attn'
    
            # Decode the text from this layer. 
            # Following this process: https://github.com/huggingface/transformers/blob/v4.46.0/src/transformers/models/llama/modeling_llama.py#L693
            hidden_states = intermediate_outputs[layer_name][0][0]
            if residual is not None:
                hidden_states = hidden_states + residual
            
            residual = hidden_states
            hidden_states = layer.post_attention_layernorm(hidden_states)
            hidden_states = layer.mlp(hidden_states)
            hidden_states = residual + hidden_states
            
            # Propagate decoder output through the LLaMA RMS Norm and the lm_head
            hidden_states = model.model.norm(hidden_states) # LLaMARMSNorm
            logits = model.lm_head(hidden_states)[0] # element 0 because we have batch size 1
            # Note: number of logits restricted to 20; this is an inherent LLM output context window limit. 
            # See this thread to understand why: https://www.reddit.com/r/LocalLLaMA/comments/1f3s0qc/why_do_llms_have_output_limit/ 
            outputs = torch.argmax(logits, dim=-1) # TODO check if this is right
            # Decode the logit - just a single token - and add to the intermediate outputs
            intermediate_text[idx] = intermediate_text[idx] + tokenizer.decode(outputs, skip_special_tokens=True)
    
            # Update residual
            residual = hidden_states

    return intermediate_text

In [24]:
# Put it all together; generate a single token output from a fixed layer
logits = get_next_logits(model, inputs['input_ids'], exit_layer=7)
next_token = logits.argmax(dim=-1)
print('Exit from layer 7; argmax ||', tokenizer.decode(next_token[0]), '\n')
next_token, _ = decode_next_token(logits, sample=True)
print('Exit from layer 7; sampling over softmax ||', tokenizer.decode(next_token[0]), '\n')

logits = get_next_logits(model, inputs['input_ids'], exit_layer=15)
next_token = logits.argmax(dim=-1)
print('Exit from layer 15; argmax ||', tokenizer.decode(next_token[0]), '\n')
next_token, _ = decode_next_token(logits, sample=True)
print('Exit from layer 15; sampling over softmax ||', tokenizer.decode(next_token[0]), '\n')

Exit from layer 7; argmax || #on in money public and not most cause. the failure system failure. according to a report report 

Exit from layer 7; sampling over softmax || <?on in corruption state’ the biggest concern behind the failure bill failure.
 and to a report report 

Exit from layer 15; argmax || #on in the use of a most cause in the delay overcrow poor. according to the report report 

Exit from layer 15; sampling over softmax || Tagsruption is public sale of being main cause
 the poor overcrow poor.
 but to a former study 



In [217]:
intermediate_text = get_all_text(model, inputs, 50, tokenizer)
for layer_number in intermediate_text:
    text = intermediate_text[layer_number]
    print('- Output from layer', layer_number, ':', text)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{'input_ids': tensor([[128000,  10803,  14455,  16239,    279,  33840,    374,    279,  10388,
          75283,    369,    279,   9527,    753,   5435,     11,   4184,    311,
            264,   3293]], device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='mps:0')}


ValueError: Input length of input_ids is 20, but `max_length` is set to 1. This can lead to unexpected behavior. You should consider increasing `max_length` or, better yet, setting `max_new_tokens`.

In [10]:
print(intermediate_text.keys())

dict_keys(['model.layers.0.self_attn.o_proj', 'model.layers.0.post_attention_layernorm', 'model.layers.1.self_attn.o_proj', 'model.layers.1.post_attention_layernorm', 'model.layers.2.self_attn.o_proj', 'model.layers.2.post_attention_layernorm', 'model.layers.3.self_attn.o_proj', 'model.layers.3.post_attention_layernorm', 'model.layers.4.self_attn.o_proj', 'model.layers.4.post_attention_layernorm', 'model.layers.5.self_attn.o_proj', 'model.layers.5.post_attention_layernorm', 'model.layers.6.self_attn.o_proj', 'model.layers.6.post_attention_layernorm', 'model.layers.7.self_attn.o_proj', 'model.layers.7.post_attention_layernorm', 'model.layers.8.self_attn.o_proj', 'model.layers.8.post_attention_layernorm', 'model.layers.9.self_attn.o_proj', 'model.layers.9.post_attention_layernorm', 'model.layers.10.self_attn.o_proj', 'model.layers.10.post_attention_layernorm', 'model.layers.11.self_attn.o_proj', 'model.layers.11.post_attention_layernorm', 'model.layers.12.self_attn.o_proj', 'model.layers

In [40]:
print(f"Generated Text (full model): {generated_text}", '\n')

# Check that generated text equals decoded text from final layer. 
# Define a hook function to capture the output of a specific layer
final_layer_embedding = {}
def hook_fn(module, input, output):       
    final_layer_embedding['a'] = output

hook_handle = model.model.norm.register_forward_hook(hook_fn)
# Perform a forward pass on the already-tokenized sample prompt
with torch.no_grad():
    out = model.generate(**inputs, max_length=50)

# Remove the hook
hook_handle.remove()

final_layer_embedding = final_layer_embedding['a']

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Generated Text (full model): Corruption involving the contractors is the chief culprit for the prison’s problems, according to a recent report by the Ministry of Public Security.
The report, released last month, says the prison has been plagued by corruption since the 1990s and has become a major target for organized crime.
In a report released last month, the Ministry of Public 



In [54]:
# Decode text, one token at a time. 
unembedding_matrix = model.get_output_embeddings().weight
predicted_token_ids = []
for k in range(20):
    output_tensor = final_layer_embedding
    # Apply the unembedding matrix
    logits = torch.matmul(output_tensor, unembedding_matrix.T)
    # Note: the model still produces sequence lengths of 20 in intermediate generations.
    # Select just the first one. 
    next_token_id = torch.argmax(logits, dim=-1)[0]
    # Break if the token is EOS
    if next_token_id == tokenizer.eos_token_id:
        break
    # Otherwise append to the predictions
    predicted_token_ids.append(next_token_id)
                    
predicted_text = tokenizer.decode(torch.as_tensor(predicted_token_ids), skip_special_tokens=True)
print('Decoded final layer output:', predicted_text)

Decoded final layer output:  in in in in in in in in in in in in in in in in in in in in


In [ ]:
# Compute averages over these scores per layer to plot.
average_scores = {}
for layer_type in real_toxicity_scores:
    average_scores[layer_type] = []
    for layer_name in real_toxicity_scores[layer_type]:
        # print(layer_name)
        # TODO Note - this depends on the layers being in order - fix this later!!
        average_scores[layer_type].append(sum(real_toxicity_scores[layer_type][layer_name]) / limit)

In [ ]:
# Plot toxicity scores vs layer depth. 
def plot_score_vs_layer(scores, layers, score_labels, attention_ax, fc_ax):
    for score_label in score_labels:
        attention_ax.plot(layers, scores['o_proj'], label=score_label)
        fc_ax.plot(layers, scores['post_attention_layernorm'], label=score_label)
        
    attention_ax.set_title('Attention Output')
    attention_ax.legend()
    fc_ax.set_title('FC Output')
    fc_ax.legend()


fig, (attention_ax, fc_ax) = plt.subplots(1, 2)
plot_score_vs_layer(average_scores, [i for i in range(16)], ['detoxify'], attention_ax, fc_ax)
plt.tight_layout()
plt.show()

In [19]:
# Do the same for other loss scores. 
trivia_qa_dataset = load_dataset("mandarjoshi/trivia_qa", "rc")['train']
trivia_qa_dataset

# Compute scores. Note: instead of detoxify toxicity score, we should use the ground truth given with the dataset. 

Generating test split: 100%|██████████| 17210/17210 [00:02<00:00, 8422.65 examples/s] 


Dataset({
    features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
    num_rows: 138384
})

In [ ]:
# Implement ACTUAL early-exiting. I.e. actually stop execution after the first layer where we exceed lambda. 
# Set arbitrary lambda threshold on the softmax output from unembedding layer.

In [ ]:
# Implement searching for a good lambda threshold that meets certain statistical requirements. 
# lambda = 0 means always early exit at the first layer, lambda=1 means never early exit.
# To get relative losses, just compute the difference between each column and the last column. 

# k = number of lambda values to search over
candidate_lambda = [i / 100 for i in range(0, 101)]
k = len(candidate_lambda)
# n_cal = number of data points in validation set. 
n_cal = len(toxicity_prompts_dataset)
# Create matrix of number of validation points. 
loss_grid = torch.zeros(k, n_cal)

for l_idx in range(k):
    l = candidate_lambda[l_idx]
    for prompt_idx in range(n_cal):
        prompt = toxicity_prompts_dataset['train'][prompt_idx]['prompt']['text']
        # TODO: run this prompt through the model with early-exit controlled by l. Get the output. 
        # Compute loss and add to matrix. This should be the toxicity score of the output from the early-exit model. 
        loss = 1
        loss_grid[l_idx, prompt_idx] = loss

In [ ]:
# Compare early-exiting performance with baseline (full-model) performance. 